In [1]:
import os, random, argparse, itertools, numpy as np, scipy.io as sio
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score, silhouette_score
from sklearn.cluster import KMeans
from sklearn.preprocessing import scale
from _utils import MMDataset, clustering_acc, overall_performance_report

class DUCMME(nn.Module):
    def __init__(self, embed_dim=200, num_samples=10000, num_views=3, feature_dims=[1000, 1000, 500], hidden_dims=[512, 512, 512], n_clusters=10, alpha=1.0):
        super(DUCMME, self).__init__()
        self.embed_dim = embed_dim; self.num_samples = num_samples; self.num_views = num_views; self.feature_dims = feature_dims; self.hidden_dims = hidden_dims; self.n_clusters = n_clusters; self.alpha = alpha
        # 1. Multi-view Feature Extraction by Fusion-Net
        self.fusion_net_encoder = nn.ModuleList([nn.Sequential(nn.Linear(feature_dims[i], hidden_dims[i]), nn.BatchNorm1d(hidden_dims[i]), nn.ReLU(),
                                                               nn.Linear(hidden_dims[i], embed_dim)) for i in range(num_views)]) # encode each view
        self.fusion_net_mha = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=10, batch_first=True) # batch_first=True: (batch_size, seq_len, hidden_dim)
        self.fusion_net_linear = nn.Linear(3*embed_dim, embed_dim) # linear projection of the fused encoded features
        # 2. Uncertainty-Aware Reconstruction by Reconstruction-Net and Uncertainty-Net
        self.reconstruct_net_list = nn.ModuleList([nn.Sequential(nn.Linear(self.embed_dim, hidden_dims[i]), nn.BatchNorm1d(hidden_dims[i]), nn.ReLU(), 
                                                                 nn.Linear(hidden_dims[i], feature_dims[i])) for i in range(num_views)]) # reconstruct each view
        self.uncertainty_net_list = nn.ModuleList([nn.Sequential(nn.Linear(self.embed_dim, hidden_dims[i]), nn.BatchNorm1d(hidden_dims[i]), nn.ReLU(), 
                                                                 nn.Linear(hidden_dims[i], feature_dims[i])) for i in range(num_views)]) # predict uncertainty for each view
        # 3. Deep Embedding Clustering by DEC
        self._cluster_centers = nn.Parameter(torch.Tensor(self.n_clusters, self.embed_dim))
        nn.init.xavier_uniform_(self._cluster_centers.data)
        
    def forward_embedding(self, x):
        encoded_output_list = [self.fusion_net_encoder[i](x[i]) for i in range(self.num_views)] # encode each view
        encoded_output_list = torch.stack(encoded_output_list, dim=1) # stack the encoded features from all views, (batch_size, num_views, embed_dim)
        encoded_output_list, _ = self.fusion_net_mha(encoded_output_list, encoded_output_list, encoded_output_list) # fuse the encoded features from all views by a multihead attention, (batch_size, num_views, embed_dim)
        encoded_output_list = encoded_output_list.contiguous().view(encoded_output_list.shape[0], -1) # flatten the encoded features, (batch_size, num_views*embed_dim)
        embedding = self.fusion_net_linear(encoded_output_list) # linear projection of the fused encoded features
        return embedding # get the embedding of the latent space H, (batch_size, embed_dim)

    def forward_uncertainty_aware_reconstruction(self, x):
        embedding = self.forward_embedding(x) # shape: [batch_size, embed_dim]
        reconstructions = [self.reconstruct_net_list[i](embedding) for i in range(self.num_views)] # reconstruct each view
        uncertainties = [self.uncertainty_net_list[i](embedding) for i in range(self.num_views)] # predict uncertainty for each view
        return reconstructions, uncertainties
        
    def forward_similarity_matrix_q(self, x): # calculate the similarity matrix q using t-distribution
        embedding = self.forward_embedding(x) # shape: [batch_size, embed_dim]
        q = 1.0 / (1.0 + torch.sum((embedding.unsqueeze(1) - self._cluster_centers) ** 2, dim=2) / self.alpha) # shape: [batch_size, n_clusters]
        q = q ** ((self.alpha + 1.0) / 2.0) # , shape: [batch_size, n_clusters]
        q = q / torch.sum(q, dim=1, keepdim=True) # Normalize q to sum to 1 across clusters, shape: [batch_size, n_clusters]
        return q, embedding # q can be regarded as the probability of the sample belonging to each cluster
    
    @property
    def cluster_centers(self):
        return self._cluster_centers.data.detach().cpu().numpy() # shape: (n_clusters, embed_dim)
    
    @cluster_centers.setter
    def cluster_centers(self, centers): # shape: (n_clusters, embed_dim)
        centers = torch.tensor(centers, dtype=torch.float32, device=self._cluster_centers.device)
        self._cluster_centers.data.copy_(centers) # copy the cluster centers to the model, set the cluster centers to the new cluster centers
        
    @staticmethod
    def target_distribution(q):
        weight = q ** 2 / torch.sum(q, dim=0) # shape: [batch_size, n_clusters]
        p = weight / torch.sum(weight, dim=1, keepdim=True) # Normalize p to sum to 1 across clusters, shape: [batch_size, n_clusters]
        return p.clone().detach()
    
    def reconstruction_loss(self, x):
        x_rec, _ = self.forward_uncertainty_aware_reconstruction(x) # reconstruct each view and predict uncertainty
        return sum([F.mse_loss(x_rec[v], x[v], reduction='mean') for v in range(self.num_views)]) # sum the losses from all views
    
    def uncertainty_aware_reconstruction_loss(self, x):
        x_rec, log_sigma_2 = self.forward_uncertainty_aware_reconstruction(x) # reconstruct each view and predict uncertainty
        # Clip log_sigma_2 to prevent numerical instability (exp(-log_sigma_2) overflow/underflow)
        # Clamp to reasonable range: -10 to 10, which corresponds to sigma^2 from exp(-10) to exp(10)
        # log_sigma_2 = [torch.clamp(log_sigma_2[v], min=-10.0, max=10.0) for v in range(self.num_views)] # shape: [num_views, batch_size, feature_dim] for numerically stable computation
        return sum([0.5 * torch.mean((x_rec[v] - x[v])**2 * torch.exp(-log_sigma_2[v]) + log_sigma_2[v]) for v in range(self.num_views)]) # uncertainty is equal to log_sigma_2
    
    def clustering_loss(self, x, p):
        q, _ = self.forward_similarity_matrix_q(x) # shape: [batch_size, n_clusters]
        return F.kl_div(q.log(), p, reduction='batchmean') # shape: ()

In [3]:
random.seed(0); np.random.seed(0); torch.manual_seed(0); torch.cuda.manual_seed_all(0) # Set random seed for reproducibility
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dataset = MMDataset('./data/data_sc_multiomics/DOGMA/', concat_data=False)
data = [x.clone().to(device) for x in dataset.X]; label = dataset.Y.clone().numpy()
data_views = dataset.data_views; data_samples = dataset.data_samples; data_features = dataset.data_features; data_categories = dataset.categories
label_dict = dataset.get_label_to_name()

dataloader = torch.utils.data.DataLoader(dataset, batch_size=data_samples, shuffle=True)
model = DUCMME(embed_dim=20, feature_dims=data_features, num_views=data_views, hidden_dims=[512, 512, 512], num_samples=data_samples, n_clusters=data_categories, alpha=1.0).to(device)
## === Stage 1: Uncertainty-Aware Reconstruction Pretraining ===
print("\n=== Stage 1: Uncertainty-Aware Reconstruction Pretraining ===")
print("Basic reconstruction training...")
optimizer = torch.optim.Adam(model.parameters(), lr=5e-3)
losses_convergence_plot_reconstruction = []
for epoch in range(200):
    model.train()
    losses = []
    for batch_idx, (x, y, idx) in enumerate(dataloader):
        x = [x.to(device) for x in x]
        optimizer.zero_grad()
        loss = model.reconstruction_loss(x)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    losses_convergence_plot_reconstruction.append(np.mean(losses))
    print(f'Epoch {epoch} completed. Average Loss: {np.mean(losses):.4f}')
embedding = model.forward_embedding(data).detach().cpu().numpy() # shape: [num_samples, embed_dim]
kmeans = KMeans(n_clusters=data_categories, n_init=20, random_state=0)
preds = kmeans.fit_predict(embedding)
acc_val = clustering_acc(label, preds)
nmi_val = normalized_mutual_info_score(label, preds)
asw_val = silhouette_score(embedding, preds)
ari_val = adjusted_rand_score(label, preds)
print(f"Pretraining completed. ACC: {acc_val:.4f}, NMI: {nmi_val:.4f}, ASW: {asw_val:.4f}, ARI: {ari_val:.4f}")
print("Uncertainty-aware reconstruction finetuning...")
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
losses_convergence_plot_reconstruction_finetuning = []
for epoch in range(100):
    model.train()
    losses = []
    for batch_idx, (x, y, idx) in enumerate(dataloader):
        x = [x.to(device) for x in x]
        optimizer.zero_grad()
        loss = model.uncertainty_aware_reconstruction_loss(x)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    losses_convergence_plot_reconstruction_finetuning.append(np.mean(losses))
    print(f'Epoch {epoch} completed. Average Loss: {np.mean(losses):.4f}')
embedding = model.forward_embedding(data).detach().cpu().numpy() # shape: [num_samples, embed_dim]
kmeans = KMeans(n_clusters=data_categories, n_init=20, random_state=0)
preds = kmeans.fit_predict(embedding)
acc_val = clustering_acc(label, preds)
nmi_val = normalized_mutual_info_score(label, preds)
asw_val = silhouette_score(embedding, preds)
ari_val = adjusted_rand_score(label, preds)
print(f"Finetuning completed. ACC: {acc_val:.4f}, NMI: {nmi_val:.4f}, ASW: {asw_val:.4f}, ARI: {ari_val:.4f}")

## === Stage 2: Deep Embedding Clustering by DEC ===
print("\n=== Stage 2: Deep Embedding Clustering ===")
print("Cluster center initialization...")
model.eval()
embedding = model.forward_embedding(data).detach().cpu().numpy() # shape: [num_samples, embed_dim]
kmeans = KMeans(n_clusters=data_categories, n_init=20, random_state=0)
preds = kmeans.fit_predict(embedding)
acc_val = clustering_acc(label, preds)
nmi_val = normalized_mutual_info_score(label, preds)
asw_val = silhouette_score(embedding, preds)
ari_val = adjusted_rand_score(label, preds)
print(f"Cluster center initialization completed. ACC: {acc_val:.4f}, NMI: {nmi_val:.4f}, ASW: {asw_val:.4f}, ARI: {ari_val:.4f}")
model.cluster_centers = kmeans.cluster_centers_ # shape: (n_clusters, embed_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.9)
losses = []
acc_convergence_plot = []
nmi_convergence_plot = []
asw_convergence_plot = []
ari_convergence_plot = []
embeddings_dict = {}
preds_dict = {}
for epoch in range(150):
    # Update target distribution periodically
    if epoch % 1 == 0:
        model.eval()
        with torch.no_grad():
            q, embedding = model.forward_similarity_matrix_q(data)
            p = model.target_distribution(q)
        y_pred = torch.argmax(q, dim=1).cpu().numpy()
        acc_val = clustering_acc(label, y_pred)
        nmi_val = normalized_mutual_info_score(label, y_pred)
        asw_val = silhouette_score(embedding.cpu().numpy(), y_pred)
        ari_val = adjusted_rand_score(label, y_pred)
        acc_convergence_plot.append(acc_val)
        nmi_convergence_plot.append(nmi_val)
        asw_convergence_plot.append(asw_val)
        ari_convergence_plot.append(ari_val)
        embeddings_dict[epoch] = embedding
        preds_dict[epoch] = y_pred
        if epoch == 0:
            delta_label = 1.0
            y_pred_last = y_pred.copy()
            print(f'[Epoch {epoch}] ACC: {acc_val:.4f}, NMI: {nmi_val:.4f}, ASW: {asw_val:.4f}, ARI: {ari_val:.4f}, Delta: {delta_label:.4f}')
        else:
            delta_label = np.sum(y_pred != y_pred_last).astype(np.float32) / y_pred.shape[0]
            y_pred_last = y_pred.copy()
            print(f'[Epoch {epoch}] ACC: {acc_val:.4f}, NMI: {nmi_val:.4f}, ASW: {asw_val:.4f}, ARI: {ari_val:.4f}, Delta: {delta_label:.4f}')
            if delta_label < 0.001:
                convergence_epoch = epoch
                print('Converged, stopping training...')
                # break
    # Training based on the target distribution that is updated periodically
    model.train()
    losses = []
    for batch_idx, (x, y, idx) in enumerate(dataloader):
        x = [x.to(device) for x in x]
        optimizer.zero_grad()
        loss = model.clustering_loss(x, p[idx])
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    scheduler.step()
    print(f'Epoch {epoch} completed. Average Loss: {np.mean(losses):.4f}')
print(f'Final ACC: {clustering_acc(label, y_pred):.4f}')

modality_rna shape: (13763, 50)
modality_protein shape: (13763, 210)
modality_atac shape: (13763, 30)

=== Stage 1: Uncertainty-Aware Reconstruction Pretraining ===
Basic reconstruction training...
Epoch 0 completed. Average Loss: 1.1189
Epoch 1 completed. Average Loss: 1.5484
Epoch 2 completed. Average Loss: 0.7363
Epoch 3 completed. Average Loss: 0.4680
Epoch 4 completed. Average Loss: 0.6113
Epoch 5 completed. Average Loss: 0.5152
Epoch 6 completed. Average Loss: 0.3349
Epoch 7 completed. Average Loss: 0.2698
Epoch 8 completed. Average Loss: 0.2938
Epoch 9 completed. Average Loss: 0.2863
Epoch 10 completed. Average Loss: 0.2275
Epoch 11 completed. Average Loss: 0.1698
Epoch 12 completed. Average Loss: 0.1492
Epoch 13 completed. Average Loss: 0.1525
Epoch 14 completed. Average Loss: 0.1536
Epoch 15 completed. Average Loss: 0.1447
Epoch 16 completed. Average Loss: 0.1260
Epoch 17 completed. Average Loss: 0.1016
Epoch 18 completed. Average Loss: 0.0850
Epoch 19 completed. Average Loss:

In [ ]:
# modality_rna shape: (13763, 50)
# modality_protein shape: (13763, 210)
# modality_atac shape: (13763, 30)
# === Stage 1: Uncertainty-Aware Reconstruction Pretraining ===
# Basic reconstruction training...
# Pretraining completed. ACC: 0.4266, NMI: 0.5018, ASW: 0.2650, ARI: 0.3690
# Uncertainty-aware reconstruction finetuning...
# Finetuning completed. ACC: 0.3837, NMI: 0.4740, ASW: 0.2247, ARI: 0.3254
# === Stage 2: Deep Embedding Clustering ===
# Cluster center initialization...
# Cluster center initialization completed. ACC: 0.3852, NMI: 0.4700, ASW: 0.2170, ARI: 0.3268
# [Epoch 0] ACC: 0.3852, NMI: 0.4700, ASW: 0.2170, ARI: 0.3268, Delta: 1.0000
# Epoch 0 completed. Average Loss: 0.2499
# [Epoch 1] ACC: 0.4048, NMI: 0.4756, ASW: 0.2075, ARI: 0.3155, Delta: 0.3549
# Epoch 1 completed. Average Loss: 0.2278
# [Epoch 2] ACC: 0.4101, NMI: 0.4781, ASW: 0.2096, ARI: 0.3164, Delta: 0.1731
# Epoch 2 completed. Average Loss: 0.2408
# [Epoch 3] ACC: 0.4000, NMI: 0.4762, ASW: 0.2105, ARI: 0.3237, Delta: 0.2381
# Epoch 3 completed. Average Loss: 0.2314
# [Epoch 4] ACC: 0.5019, NMI: 0.4833, ASW: 0.1998, ARI: 0.4052, Delta: 0.2390
# Epoch 4 completed. Average Loss: 0.2046
# [Epoch 5] ACC: 0.5625, NMI: 0.4979, ASW: 0.2036, ARI: 0.4956, Delta: 0.1846
# Epoch 5 completed. Average Loss: 0.1806
# [Epoch 6] ACC: 0.5588, NMI: 0.5015, ASW: 0.2173, ARI: 0.4994, Delta: 0.1197
# Epoch 6 completed. Average Loss: 0.1657
# [Epoch 7] ACC: 0.5420, NMI: 0.4924, ASW: 0.2220, ARI: 0.4489, Delta: 0.1182
# Epoch 7 completed. Average Loss: 0.1606
# [Epoch 8] ACC: 0.5144, NMI: 0.4816, ASW: 0.2104, ARI: 0.3888, Delta: 0.1413
# Epoch 8 completed. Average Loss: 0.1613
# [Epoch 9] ACC: 0.5016, NMI: 0.4810, ASW: 0.1977, ARI: 0.3811, Delta: 0.1574
# Epoch 9 completed. Average Loss: 0.1632
# [Epoch 10] ACC: 0.5527, NMI: 0.4840, ASW: 0.1893, ARI: 0.4307, Delta: 0.1524
# Epoch 10 completed. Average Loss: 0.1653
# [Epoch 11] ACC: 0.5471, NMI: 0.4774, ASW: 0.1841, ARI: 0.4216, Delta: 0.1447
# Epoch 11 completed. Average Loss: 0.1707
# [Epoch 12] ACC: 0.4871, NMI: 0.4649, ASW: 0.1684, ARI: 0.3287, Delta: 0.1646
# Epoch 12 completed. Average Loss: 0.1825
# [Epoch 13] ACC: 0.4083, NMI: 0.4578, ASW: 0.1467, ARI: 0.2851, Delta: 0.1907
# Epoch 13 completed. Average Loss: 0.1976
# [Epoch 14] ACC: 0.4640, NMI: 0.4684, ASW: 0.1421, ARI: 0.3587, Delta: 0.1955
# Epoch 14 completed. Average Loss: 0.2101
# [Epoch 15] ACC: 0.4889, NMI: 0.4785, ASW: 0.1632, ARI: 0.4327, Delta: 0.1905
# Epoch 15 completed. Average Loss: 0.2141
# [Epoch 16] ACC: 0.5149, NMI: 0.4851, ASW: 0.1986, ARI: 0.4714, Delta: 0.2003
# Epoch 16 completed. Average Loss: 0.2142
# [Epoch 17] ACC: 0.5223, NMI: 0.4842, ASW: 0.3056, ARI: 0.4740, Delta: 0.1971
# Epoch 17 completed. Average Loss: 0.2147
# [Epoch 18] ACC: 0.5126, NMI: 0.4754, ASW: 0.2646, ARI: 0.4260, Delta: 0.2172
# Epoch 18 completed. Average Loss: 0.2211
# [Epoch 19] ACC: 0.5096, NMI: 0.4638, ASW: 0.2104, ARI: 0.3618, Delta: 0.2289
# Epoch 19 completed. Average Loss: 0.2305
# [Epoch 20] ACC: 0.4793, NMI: 0.4566, ASW: 0.1778, ARI: 0.3471, Delta: 0.2430
# Epoch 20 completed. Average Loss: 0.2424
# [Epoch 21] ACC: 0.5428, NMI: 0.4603, ASW: 0.1194, ARI: 0.4219, Delta: 0.2185
# Epoch 21 completed. Average Loss: 0.2533
# [Epoch 22] ACC: 0.5748, NMI: 0.4535, ASW: 0.0749, ARI: 0.4354, Delta: 0.1828
# Epoch 22 completed. Average Loss: 0.2634
# [Epoch 23] ACC: 0.5045, NMI: 0.4180, ASW: 0.0260, ARI: 0.3177, Delta: 0.1798
# Epoch 23 completed. Average Loss: 0.2637
# [Epoch 24] ACC: 0.4062, NMI: 0.3778, ASW: -0.0215, ARI: 0.2356, Delta: 0.1932
# Epoch 24 completed. Average Loss: 0.2533
# [Epoch 25] ACC: 0.3804, NMI: 0.3604, ASW: -0.0364, ARI: 0.2548, Delta: 0.1624
# Epoch 25 completed. Average Loss: 0.2354
# [Epoch 26] ACC: 0.4034, NMI: 0.3519, ASW: -0.0003, ARI: 0.2744, Delta: 0.1118
# Epoch 26 completed. Average Loss: 0.2098
# [Epoch 27] ACC: 0.4413, NMI: 0.3420, ASW: 0.0530, ARI: 0.2780, Delta: 0.0945
# Epoch 27 completed. Average Loss: 0.1766
# [Epoch 28] ACC: 0.4729, NMI: 0.3398, ASW: 0.1249, ARI: 0.2806, Delta: 0.0817
# Epoch 28 completed. Average Loss: 0.1421
# [Epoch 29] ACC: 0.4946, NMI: 0.3379, ASW: 0.1647, ARI: 0.2870, Delta: 0.0769
# Epoch 29 completed. Average Loss: 0.1122
# [Epoch 30] ACC: 0.5131, NMI: 0.3382, ASW: 0.1629, ARI: 0.2916, Delta: 0.0776
# Epoch 30 completed. Average Loss: 0.0878
# [Epoch 31] ACC: 0.5248, NMI: 0.3326, ASW: 0.1801, ARI: 0.2864, Delta: 0.0801
# Epoch 31 completed. Average Loss: 0.0678
# [Epoch 32] ACC: 0.5287, NMI: 0.3258, ASW: 0.1751, ARI: 0.2746, Delta: 0.0848
# Epoch 32 completed. Average Loss: 0.0524
# [Epoch 33] ACC: 0.5221, NMI: 0.3167, ASW: 0.1546, ARI: 0.2487, Delta: 0.0850
# Epoch 33 completed. Average Loss: 0.0414
# [Epoch 34] ACC: 0.5074, NMI: 0.3089, ASW: 0.1293, ARI: 0.2106, Delta: 0.0871
# Epoch 34 completed. Average Loss: 0.0339
# [Epoch 35] ACC: 0.4814, NMI: 0.2982, ASW: 0.0942, ARI: 0.1683, Delta: 0.0834
# Epoch 35 completed. Average Loss: 0.0286
# [Epoch 36] ACC: 0.4479, NMI: 0.2872, ASW: 0.0631, ARI: 0.1225, Delta: 0.0889
# Epoch 36 completed. Average Loss: 0.0244
# [Epoch 37] ACC: 0.4085, NMI: 0.2791, ASW: 0.0395, ARI: 0.0865, Delta: 0.0922
# Epoch 37 completed. Average Loss: 0.0209
# [Epoch 38] ACC: 0.3737, NMI: 0.2699, ASW: -0.0186, ARI: 0.0625, Delta: 0.0829
# Epoch 38 completed. Average Loss: 0.0179
# [Epoch 39] ACC: 0.3561, NMI: 0.2583, ASW: -0.0306, ARI: 0.0499, Delta: 0.0769
# Epoch 39 completed. Average Loss: 0.0155
# [Epoch 40] ACC: 0.3550, NMI: 0.2555, ASW: -0.0413, ARI: 0.0492, Delta: 0.0692
# Epoch 40 completed. Average Loss: 0.0135
# [Epoch 41] ACC: 0.3520, NMI: 0.2518, ASW: -0.0444, ARI: 0.0527, Delta: 0.0442
# Epoch 41 completed. Average Loss: 0.0121
# [Epoch 42] ACC: 0.3458, NMI: 0.2519, ASW: -0.0517, ARI: 0.0589, Delta: 0.0410
# Epoch 42 completed. Average Loss: 0.0109
# [Epoch 43] ACC: 0.3369, NMI: 0.2469, ASW: -0.0531, ARI: 0.0651, Delta: 0.0318
# Epoch 43 completed. Average Loss: 0.0100
# [Epoch 44] ACC: 0.3302, NMI: 0.2452, ASW: -0.0356, ARI: 0.0725, Delta: 0.0260
# Epoch 44 completed. Average Loss: 0.0092
# [Epoch 45] ACC: 0.3218, NMI: 0.2436, ASW: -0.0359, ARI: 0.0788, Delta: 0.0201
# Epoch 45 completed. Average Loss: 0.0085
# [Epoch 46] ACC: 0.3148, NMI: 0.2408, ASW: -0.0341, ARI: 0.0840, Delta: 0.0155
# Epoch 46 completed. Average Loss: 0.0080
# [Epoch 47] ACC: 0.3180, NMI: 0.2398, ASW: -0.0273, ARI: 0.0891, Delta: 0.0129
# Epoch 47 completed. Average Loss: 0.0075
# [Epoch 48] ACC: 0.3243, NMI: 0.2392, ASW: -0.0239, ARI: 0.0936, Delta: 0.0097
# Epoch 48 completed. Average Loss: 0.0071
# [Epoch 49] ACC: 0.3300, NMI: 0.2388, ASW: -0.0276, ARI: 0.0980, Delta: 0.0084
# Epoch 49 completed. Average Loss: 0.0068
# [Epoch 50] ACC: 0.3353, NMI: 0.2380, ASW: -0.0178, ARI: 0.1021, Delta: 0.0081
# Epoch 50 completed. Average Loss: 0.0065
# [Epoch 51] ACC: 0.3391, NMI: 0.2375, ASW: -0.0195, ARI: 0.1052, Delta: 0.0062
# Epoch 51 completed. Average Loss: 0.0062
# [Epoch 52] ACC: 0.3427, NMI: 0.2385, ASW: -0.0101, ARI: 0.1081, Delta: 0.0058
# Epoch 52 completed. Average Loss: 0.0060
# [Epoch 53] ACC: 0.3451, NMI: 0.2380, ASW: -0.0098, ARI: 0.1099, Delta: 0.0049
# Epoch 53 completed. Average Loss: 0.0059
# [Epoch 54] ACC: 0.3473, NMI: 0.2376, ASW: -0.0078, ARI: 0.1120, Delta: 0.0047
# Epoch 54 completed. Average Loss: 0.0057
# [Epoch 55] ACC: 0.3490, NMI: 0.2380, ASW: -0.0025, ARI: 0.1145, Delta: 0.0042
# Epoch 55 completed. Average Loss: 0.0056
# [Epoch 56] ACC: 0.3501, NMI: 0.2367, ASW: 0.0047, ARI: 0.1156, Delta: 0.0031
# Epoch 56 completed. Average Loss: 0.0055
# [Epoch 57] ACC: 0.3513, NMI: 0.2375, ASW: 0.0076, ARI: 0.1172, Delta: 0.0038
# Epoch 57 completed. Average Loss: 0.0055
# [Epoch 58] ACC: 0.3522, NMI: 0.2382, ASW: 0.0026, ARI: 0.1182, Delta: 0.0027
# Epoch 58 completed. Average Loss: 0.0054
# [Epoch 59] ACC: 0.3525, NMI: 0.2370, ASW: 0.0031, ARI: 0.1190, Delta: 0.0029
# Epoch 59 completed. Average Loss: 0.0054
# [Epoch 60] ACC: 0.3536, NMI: 0.2377, ASW: 0.0054, ARI: 0.1201, Delta: 0.0031
# Epoch 60 completed. Average Loss: 0.0054
# [Epoch 61] ACC: 0.3538, NMI: 0.2369, ASW: 0.0058, ARI: 0.1203, Delta: 0.0017
# Epoch 61 completed. Average Loss: 0.0054
# [Epoch 62] ACC: 0.3541, NMI: 0.2366, ASW: 0.0039, ARI: 0.1212, Delta: 0.0023
# Epoch 62 completed. Average Loss: 0.0055
# [Epoch 63] ACC: 0.3543, NMI: 0.2366, ASW: 0.0016, ARI: 0.1214, Delta: 0.0012
# Epoch 63 completed. Average Loss: 0.0056
# [Epoch 64] ACC: 0.3544, NMI: 0.2362, ASW: 0.0011, ARI: 0.1216, Delta: 0.0014
# Epoch 64 completed. Average Loss: 0.0056
# [Epoch 65] ACC: 0.3546, NMI: 0.2363, ASW: 0.0063, ARI: 0.1223, Delta: 0.0016
# Epoch 65 completed. Average Loss: 0.0057
# [Epoch 66] ACC: 0.3549, NMI: 0.2360, ASW: 0.0050, ARI: 0.1225, Delta: 0.0017
# Epoch 66 completed. Average Loss: 0.0058
# [Epoch 67] ACC: 0.3552, NMI: 0.2364, ASW: 0.0033, ARI: 0.1228, Delta: 0.0010
# Epoch 67 completed. Average Loss: 0.0060
# [Epoch 68] ACC: 0.3556, NMI: 0.2373, ASW: 0.0027, ARI: 0.1236, Delta: 0.0013
# Epoch 68 completed. Average Loss: 0.0061
# [Epoch 69] ACC: 0.3557, NMI: 0.2368, ASW: 0.0009, ARI: 0.1238, Delta: 0.0014
# Epoch 69 completed. Average Loss: 0.0062
# [Epoch 70] ACC: 0.3558, NMI: 0.2371, ASW: -0.0016, ARI: 0.1240, Delta: 0.0009
# Converged, stopping training...
# Epoch 70 completed. Average Loss: 0.0064
# [Epoch 71] ACC: 0.3557, NMI: 0.2371, ASW: -0.0032, ARI: 0.1240, Delta: 0.0012
# Epoch 71 completed. Average Loss: 0.0065
# [Epoch 72] ACC: 0.3557, NMI: 0.2373, ASW: -0.0021, ARI: 0.1241, Delta: 0.0012
# Epoch 72 completed. Average Loss: 0.0067
# [Epoch 73] ACC: 0.3554, NMI: 0.2363, ASW: -0.0049, ARI: 0.1240, Delta: 0.0015
# Epoch 73 completed. Average Loss: 0.0069
# [Epoch 74] ACC: 0.3549, NMI: 0.2361, ASW: -0.0072, ARI: 0.1236, Delta: 0.0011
# Epoch 74 completed. Average Loss: 0.0071
# [Epoch 75] ACC: 0.3549, NMI: 0.2360, ASW: -0.0105, ARI: 0.1237, Delta: 0.0011
# Epoch 75 completed. Average Loss: 0.0073
# [Epoch 76] ACC: 0.3546, NMI: 0.2359, ASW: -0.0120, ARI: 0.1236, Delta: 0.0011
# Epoch 76 completed. Average Loss: 0.0075
# [Epoch 77] ACC: 0.3542, NMI: 0.2353, ASW: -0.0143, ARI: 0.1235, Delta: 0.0020
# Epoch 77 completed. Average Loss: 0.0078
# [Epoch 78] ACC: 0.3541, NMI: 0.2355, ASW: -0.0153, ARI: 0.1234, Delta: 0.0010
# Epoch 78 completed. Average Loss: 0.0080
# [Epoch 79] ACC: 0.3538, NMI: 0.2354, ASW: -0.0178, ARI: 0.1232, Delta: 0.0010
# Epoch 79 completed. Average Loss: 0.0083
# [Epoch 80] ACC: 0.3531, NMI: 0.2364, ASW: -0.0202, ARI: 0.1233, Delta: 0.0021
# Epoch 80 completed. Average Loss: 0.0086
# [Epoch 81] ACC: 0.3526, NMI: 0.2368, ASW: -0.0225, ARI: 0.1233, Delta: 0.0016
# Epoch 81 completed. Average Loss: 0.0089
# [Epoch 82] ACC: 0.3517, NMI: 0.2376, ASW: -0.0257, ARI: 0.1232, Delta: 0.0027
# Epoch 82 completed. Average Loss: 0.0093
# [Epoch 83] ACC: 0.3513, NMI: 0.2378, ASW: -0.0284, ARI: 0.1232, Delta: 0.0019
# Epoch 83 completed. Average Loss: 0.0096
# [Epoch 84] ACC: 0.3505, NMI: 0.2379, ASW: -0.0313, ARI: 0.1227, Delta: 0.0019
# Epoch 84 completed. Average Loss: 0.0100
# [Epoch 85] ACC: 0.3499, NMI: 0.2383, ASW: -0.0318, ARI: 0.1227, Delta: 0.0017
# Epoch 85 completed. Average Loss: 0.0104
# [Epoch 86] ACC: 0.3492, NMI: 0.2388, ASW: -0.0320, ARI: 0.1227, Delta: 0.0019
# Epoch 86 completed. Average Loss: 0.0108
# [Epoch 87] ACC: 0.3487, NMI: 0.2392, ASW: -0.0352, ARI: 0.1223, Delta: 0.0024
# Epoch 87 completed. Average Loss: 0.0112
# [Epoch 88] ACC: 0.3480, NMI: 0.2393, ASW: -0.0359, ARI: 0.1220, Delta: 0.0024
# Epoch 88 completed. Average Loss: 0.0117
# [Epoch 89] ACC: 0.3475, NMI: 0.2397, ASW: -0.0371, ARI: 0.1219, Delta: 0.0021
# Epoch 89 completed. Average Loss: 0.0121
# [Epoch 90] ACC: 0.3463, NMI: 0.2403, ASW: -0.0475, ARI: 0.1215, Delta: 0.0031
# Epoch 90 completed. Average Loss: 0.0126
# [Epoch 91] ACC: 0.3455, NMI: 0.2407, ASW: -0.0508, ARI: 0.1212, Delta: 0.0031
# Epoch 91 completed. Average Loss: 0.0132
# [Epoch 92] ACC: 0.3443, NMI: 0.2403, ASW: -0.0534, ARI: 0.1202, Delta: 0.0039
# Epoch 92 completed. Average Loss: 0.0138
# [Epoch 93] ACC: 0.3421, NMI: 0.2412, ASW: -0.0574, ARI: 0.1193, Delta: 0.0049
# Epoch 93 completed. Average Loss: 0.0144
# [Epoch 94] ACC: 0.3402, NMI: 0.2414, ASW: -0.0586, ARI: 0.1183, Delta: 0.0031
# Epoch 94 completed. Average Loss: 0.0151
# [Epoch 95] ACC: 0.3392, NMI: 0.2420, ASW: -0.0612, ARI: 0.1180, Delta: 0.0039
# Epoch 95 completed. Average Loss: 0.0158
# [Epoch 96] ACC: 0.3378, NMI: 0.2426, ASW: -0.0625, ARI: 0.1176, Delta: 0.0046
# Epoch 96 completed. Average Loss: 0.0167
# [Epoch 97] ACC: 0.3362, NMI: 0.2422, ASW: -0.0633, ARI: 0.1171, Delta: 0.0043
# Epoch 97 completed. Average Loss: 0.0177
# [Epoch 98] ACC: 0.3336, NMI: 0.2434, ASW: -0.0657, ARI: 0.1164, Delta: 0.0054
# Epoch 98 completed. Average Loss: 0.0188
# [Epoch 99] ACC: 0.3317, NMI: 0.2438, ASW: -0.0748, ARI: 0.1160, Delta: 0.0063
# Epoch 99 completed. Average Loss: 0.0202
# [Epoch 100] ACC: 0.3294, NMI: 0.2454, ASW: -0.0778, ARI: 0.1151, Delta: 0.0069
# Epoch 100 completed. Average Loss: 0.0219
# [Epoch 101] ACC: 0.3272, NMI: 0.2451, ASW: -0.0780, ARI: 0.1144, Delta: 0.0068
# Epoch 101 completed. Average Loss: 0.0238
# [Epoch 102] ACC: 0.3251, NMI: 0.2454, ASW: -0.0827, ARI: 0.1135, Delta: 0.0088
# Epoch 102 completed. Average Loss: 0.0261
# [Epoch 103] ACC: 0.3217, NMI: 0.2475, ASW: -0.0833, ARI: 0.1122, Delta: 0.0083
# Epoch 103 completed. Average Loss: 0.0288
# [Epoch 104] ACC: 0.3200, NMI: 0.2478, ASW: -0.0892, ARI: 0.1114, Delta: 0.0088
# Epoch 104 completed. Average Loss: 0.0318
# [Epoch 105] ACC: 0.3157, NMI: 0.2507, ASW: -0.0919, ARI: 0.1096, Delta: 0.0117
# Epoch 105 completed. Average Loss: 0.0353
# [Epoch 106] ACC: 0.3108, NMI: 0.2522, ASW: -0.0985, ARI: 0.1080, Delta: 0.0124
# Epoch 106 completed. Average Loss: 0.0391
# [Epoch 107] ACC: 0.3174, NMI: 0.2545, ASW: -0.1040, ARI: 0.1060, Delta: 0.0139
# Epoch 107 completed. Average Loss: 0.0432
# [Epoch 108] ACC: 0.3264, NMI: 0.2574, ASW: -0.1050, ARI: 0.1038, Delta: 0.0171
# Epoch 108 completed. Average Loss: 0.0478
# [Epoch 109] ACC: 0.3366, NMI: 0.2610, ASW: -0.0986, ARI: 0.1017, Delta: 0.0192
# Epoch 109 completed. Average Loss: 0.0528
# [Epoch 110] ACC: 0.3464, NMI: 0.2642, ASW: -0.0934, ARI: 0.1009, Delta: 0.0182
# Epoch 110 completed. Average Loss: 0.0581
# [Epoch 111] ACC: 0.3573, NMI: 0.2710, ASW: -0.0864, ARI: 0.1011, Delta: 0.0214
# Epoch 111 completed. Average Loss: 0.0631
# [Epoch 112] ACC: 0.3678, NMI: 0.2761, ASW: -0.0777, ARI: 0.1014, Delta: 0.0227
# Epoch 112 completed. Average Loss: 0.0670
# [Epoch 113] ACC: 0.3794, NMI: 0.2825, ASW: -0.0751, ARI: 0.1028, Delta: 0.0257
# Epoch 113 completed. Average Loss: 0.0696
# [Epoch 114] ACC: 0.3924, NMI: 0.2875, ASW: -0.0727, ARI: 0.1053, Delta: 0.0288
# Epoch 114 completed. Average Loss: 0.0714
# [Epoch 115] ACC: 0.4063, NMI: 0.2935, ASW: -0.0654, ARI: 0.1103, Delta: 0.0356
# Epoch 115 completed. Average Loss: 0.0727
# [Epoch 116] ACC: 0.4169, NMI: 0.2981, ASW: -0.0643, ARI: 0.1177, Delta: 0.0411
# Epoch 116 completed. Average Loss: 0.0740
# [Epoch 117] ACC: 0.4322, NMI: 0.3059, ASW: -0.0625, ARI: 0.1294, Delta: 0.0478
# Epoch 117 completed. Average Loss: 0.0749
# [Epoch 118] ACC: 0.4455, NMI: 0.3143, ASW: -0.0504, ARI: 0.1441, Delta: 0.0544
# Epoch 118 completed. Average Loss: 0.0751
# [Epoch 119] ACC: 0.4563, NMI: 0.3206, ASW: -0.0350, ARI: 0.1590, Delta: 0.0610
# Epoch 119 completed. Average Loss: 0.0745
# [Epoch 120] ACC: 0.4687, NMI: 0.3253, ASW: -0.0159, ARI: 0.1750, Delta: 0.0606
# Epoch 120 completed. Average Loss: 0.0737
# [Epoch 121] ACC: 0.4756, NMI: 0.3278, ASW: -0.0144, ARI: 0.1910, Delta: 0.0574
# Epoch 121 completed. Average Loss: 0.0729
# [Epoch 122] ACC: 0.4833, NMI: 0.3325, ASW: -0.0026, ARI: 0.2120, Delta: 0.0628
# Epoch 122 completed. Average Loss: 0.0722
# [Epoch 123] ACC: 0.4925, NMI: 0.3407, ASW: 0.0177, ARI: 0.2292, Delta: 0.0636
# Epoch 123 completed. Average Loss: 0.0716
# [Epoch 124] ACC: 0.5047, NMI: 0.3577, ASW: 0.0504, ARI: 0.2536, Delta: 0.0696
# Epoch 124 completed. Average Loss: 0.0708
# [Epoch 125] ACC: 0.5129, NMI: 0.3696, ASW: 0.0776, ARI: 0.2717, Delta: 0.0634
# Epoch 125 completed. Average Loss: 0.0702
# [Epoch 126] ACC: 0.5215, NMI: 0.3808, ASW: 0.0986, ARI: 0.2891, Delta: 0.0583
# Epoch 126 completed. Average Loss: 0.0706
# [Epoch 127] ACC: 0.5276, NMI: 0.3911, ASW: 0.1254, ARI: 0.3015, Delta: 0.0582
# Epoch 127 completed. Average Loss: 0.0719
# [Epoch 128] ACC: 0.5341, NMI: 0.3987, ASW: 0.1425, ARI: 0.3122, Delta: 0.0490
# Epoch 128 completed. Average Loss: 0.0736
# [Epoch 129] ACC: 0.5385, NMI: 0.4048, ASW: 0.1599, ARI: 0.3205, Delta: 0.0480
# Epoch 129 completed. Average Loss: 0.0757
# [Epoch 130] ACC: 0.5391, NMI: 0.4094, ASW: 0.1713, ARI: 0.3229, Delta: 0.0419
# Epoch 130 completed. Average Loss: 0.0784
# [Epoch 131] ACC: 0.5356, NMI: 0.4124, ASW: 0.1758, ARI: 0.3226, Delta: 0.0412
# Epoch 131 completed. Average Loss: 0.0817
# [Epoch 132] ACC: 0.5308, NMI: 0.4124, ASW: 0.1804, ARI: 0.3204, Delta: 0.0368
# Epoch 132 completed. Average Loss: 0.0858
# [Epoch 133] ACC: 0.5193, NMI: 0.4150, ASW: 0.1726, ARI: 0.3160, Delta: 0.0413
# Epoch 133 completed. Average Loss: 0.0906
# [Epoch 134] ACC: 0.5029, NMI: 0.4137, ASW: 0.1724, ARI: 0.3083, Delta: 0.0415
# Epoch 134 completed. Average Loss: 0.0962
# [Epoch 135] ACC: 0.4888, NMI: 0.4114, ASW: 0.1736, ARI: 0.3029, Delta: 0.0384
# Epoch 135 completed. Average Loss: 0.1020
# [Epoch 136] ACC: 0.4717, NMI: 0.4091, ASW: 0.1703, ARI: 0.2970, Delta: 0.0422
# Epoch 136 completed. Average Loss: 0.1078
# [Epoch 137] ACC: 0.4719, NMI: 0.4063, ASW: 0.1497, ARI: 0.2909, Delta: 0.0422
# Epoch 137 completed. Average Loss: 0.1136
# [Epoch 138] ACC: 0.4667, NMI: 0.4015, ASW: 0.1496, ARI: 0.2865, Delta: 0.0477
# Epoch 138 completed. Average Loss: 0.1196
# [Epoch 139] ACC: 0.4622, NMI: 0.3978, ASW: 0.1479, ARI: 0.2838, Delta: 0.0498
# Epoch 139 completed. Average Loss: 0.1263
# [Epoch 140] ACC: 0.4558, NMI: 0.3912, ASW: 0.1404, ARI: 0.2831, Delta: 0.0600
# Epoch 140 completed. Average Loss: 0.1346
# [Epoch 141] ACC: 0.4678, NMI: 0.3846, ASW: 0.1376, ARI: 0.2834, Delta: 0.0699
# Epoch 141 completed. Average Loss: 0.1433
# [Epoch 142] ACC: 0.4736, NMI: 0.3800, ASW: 0.1333, ARI: 0.2846, Delta: 0.0780
# Epoch 142 completed. Average Loss: 0.1526
# [Epoch 143] ACC: 0.4716, NMI: 0.3705, ASW: 0.1445, ARI: 0.2768, Delta: 0.0989
# Epoch 143 completed. Average Loss: 0.1611
# [Epoch 144] ACC: 0.4548, NMI: 0.3592, ASW: 0.1410, ARI: 0.2624, Delta: 0.1184
# Epoch 144 completed. Average Loss: 0.1668
# [Epoch 145] ACC: 0.4197, NMI: 0.3451, ASW: 0.1428, ARI: 0.2401, Delta: 0.1367
# Epoch 145 completed. Average Loss: 0.1676
# [Epoch 146] ACC: 0.3853, NMI: 0.3361, ASW: 0.0927, ARI: 0.2244, Delta: 0.1458
# Epoch 146 completed. Average Loss: 0.1640
# [Epoch 147] ACC: 0.3727, NMI: 0.3257, ASW: 0.0610, ARI: 0.2099, Delta: 0.1567
# Epoch 147 completed. Average Loss: 0.1568
# [Epoch 148] ACC: 0.3813, NMI: 0.3198, ASW: 0.0377, ARI: 0.1963, Delta: 0.1567
# Epoch 148 completed. Average Loss: 0.1481
# [Epoch 149] ACC: 0.3670, NMI: 0.3131, ASW: 0.0206, ARI: 0.1898, Delta: 0.1450
# Epoch 149 completed. Average Loss: 0.1413
# Final ACC: 0.3670